# Grok-multimodal · FS03-FS04 Captioning

Show&Tell (CNN+LSTM) then Show-Attend-Tell (spatial attention).


In [ ]:
import os, json, math, random, time, re
from pathlib import Path
from collections import Counter
import numpy as np
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT=Path("/kaggle/working"); FIG=OUT/"figures"; RES=OUT/"results"
FIG.mkdir(parents=True, exist_ok=True); RES.mkdir(parents=True, exist_ok=True)
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device", device, "gpus", torch.cuda.device_count() if torch.cuda.is_available() else 0)
PROGRESS={}

def make_shape_image(kind, size=64):
    img=np.ones((size,size,3),np.float32)*0.95
    yy,xx=np.mgrid[0:size,0:size]; cy,cx=size//2,size//2
    if kind=="red_circle":
        m=(yy-cy)**2+(xx-cx)**2<=(size*0.28)**2; img[m]=(0.9,0.15,0.12)
    elif kind=="blue_square":
        m=(np.abs(yy-cy)<size*0.25)&(np.abs(xx-cx)<size*0.25); img[m]=(0.15,0.25,0.85)
    elif kind=="green_triangle":
        m=(yy>cy-size*0.25)&(yy<cy+size*0.3)
        m&=np.abs(xx-cx)<(yy-(cy-size*0.25))*0.7; img[m]=(0.15,0.75,0.25)
    elif kind=="yellow_circle":
        m=(yy-cy)**2+(xx-cx)**2<=(size*0.28)**2; img[m]=(0.95,0.85,0.1)
    else:
        raise ValueError(kind)
    return img


## FS03 · Show and Tell


In [ ]:
# Show & Tell style: CNN encoder + LSTM decoder captioning
PAD, BOS, EOS, UNK = "<pad>", "<bos>", "<eos>", "<unk>"
CAPTIONS = {
    "red_circle": ["a red circle", "red round shape", "a circle that is red"],
    "blue_square": ["a blue square", "blue box shape", "a square that is blue"],
    "green_triangle": ["a green triangle", "green pointed shape", "a triangle that is green"],
}
CLASSES=list(CAPTIONS.keys())

# build vocab
counter=Counter()
for caps in CAPTIONS.values():
    for c in caps:
        counter.update(c.split())
itos=[PAD,BOS,EOS,UNK]+sorted(counter.keys())
stoi={t:i for i,t in enumerate(itos)}
V=len(itos)

def encode_cap(s, max_len=8):
    ids=[stoi[BOS]]+[stoi.get(t,stoi[UNK]) for t in s.split()]+[stoi[EOS]]
    ids=ids[:max_len]
    ids+=[stoi[PAD]]*(max_len-len(ids))
    return ids

class CapDS(Dataset):
    def __init__(self, n_per=120, size=32):
        self.items=[]
        for k in CLASSES:
            for _ in range(n_per):
                img=make_shape_image(k,64)[::2,::2][:size,:size]
                img=np.clip(img+np.random.randn(*img.shape).astype(np.float32)*0.04,0,1)
                cap=random.choice(CAPTIONS[k])
                self.items.append((img.transpose(2,0,1), encode_cap(cap), k))
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        x,y,k=self.items[i]
        return torch.tensor(x,dtype=torch.float32), torch.tensor(y,dtype=torch.long), k

class Encoder(nn.Module):
    def __init__(self, emb=128):
        super().__init__()
        self.cnn=nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,emb,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )
    def forward(self,x): return self.cnn(x)

class Decoder(nn.Module):
    def __init__(self, emb=128, hid=128):
        super().__init__()
        self.emb=nn.Embedding(V, emb, padding_idx=stoi[PAD])
        self.lstm=nn.LSTM(emb, hid, batch_first=True)
        self.fc=nn.Linear(hid, V)
        self.init=nn.Linear(emb, hid)
    def forward(self, feats, caps):
        # teacher forcing: caps [B,T]
        h0=torch.tanh(self.init(feats)).unsqueeze(0)  # [1,B,H]
        c0=torch.zeros_like(h0)
        e=self.emb(caps[:,:-1])
        # inject image at t=0 by adding to first emb
        e=e.clone(); e[:,0]=e[:,0]+feats
        out,_=self.lstm(e,(h0,c0))
        return self.fc(out)

class ShowTell(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc=Encoder(); self.dec=Decoder()
    def forward(self,x,caps):
        return self.dec(self.enc(x), caps)

ds=CapDS(); n=len(ds); idx=torch.randperm(n)
tr,va=idx[:int(0.85*n)], idx[int(0.85*n):]
def subset(ix):
    xs,ys=[],[]
    for i in ix.tolist():
        x,y,_=ds[i]; xs.append(x); ys.append(y)
    return torch.stack(xs), torch.stack(ys)
xtr,ytr=subset(tr); xva,yva=subset(va)
model=ShowTell().to(device)
opt=torch.optim.Adam(model.parameters(), lr=2e-3)
hist=[]
for epoch in range(1,12):
    model.train()
    perm=torch.randperm(len(xtr)); losses=[]
    for i in range(0,len(xtr),64):
        b=perm[i:i+64]
        xb,yb=xtr[b].to(device), ytr[b].to(device)
        opt.zero_grad(set_to_none=True)
        logits=model(xb,yb)  # [B,T-1,V]
        loss=F.cross_entropy(logits.reshape(-1,V), yb[:,1:].reshape(-1), ignore_index=stoi[PAD])
        loss.backward(); opt.step(); losses.append(loss.item())
    # val token acc
    model.eval()
    with torch.no_grad():
        lg=model(xva.to(device), yva.to(device))
        pred=lg.argmax(-1)
        mask=yva[:,1:].to(device)!=stoi[PAD]
        acc=(pred[mask]==yva[:,1:].to(device)[mask]).float().mean().item()
    row={"epoch":epoch,"loss":round(float(np.mean(losses)),4),"val_token_acc":round(acc,4)}
    hist.append(row); print(row)

@torch.no_grad()
def greedy_caption(img_kind, max_len=8):
    img=make_shape_image(img_kind,64)[::2,::2][:32,:32]
    x=torch.tensor(img.transpose(2,0,1)[None],dtype=torch.float32,device=device)
    model.eval(); feat=model.enc(x)
    tok=torch.tensor([[stoi[BOS]]],device=device)
    h=torch.tanh(model.dec.init(feat)).unsqueeze(0); c=torch.zeros_like(h)
    out_ids=[]
    for t in range(max_len-1):
        e=model.dec.emb(tok[:,-1:])
        if t==0: e=e+feat.unsqueeze(1)
        o,(h,c)=model.dec.lstm(e,(h,c))
        nxt=model.dec.fc(o[:,-1]).argmax(-1)
        if nxt.item()==stoi[EOS]: break
        out_ids.append(nxt.item()); tok=torch.cat([tok,nxt[:,None]],1)
    return " ".join(itos[i] for i in out_ids if i not in (stoi[PAD],stoi[BOS],stoi[EOS]))

samples=[]
fig,axes=plt.subplots(1,3,figsize=(9,3.2))
for ax,k in zip(axes,CLASSES):
    pred=greedy_caption(k)
    samples.append({"input":k,"pred":pred,"refs":CAPTIONS[k]})
    ax.imshow(make_shape_image(k)); ax.set_title(f"pred: {pred}",fontsize=9); ax.axis("off")
fig.suptitle("FS03 Show&Tell CNN+LSTM"); fig.tight_layout()
fig.savefig(FIG/"fs03_captions.png",dpi=120); plt.close()
print(samples)
fs03={"stage":"FS03","method":"Show-and-Tell CNN encoder + LSTM decoder",
      "history":hist,"samples":samples,
      "vs_prev":"FS02 closed-set class word; FS03 open sequence generation",
      "figure":"figures/fs03_captions.png"}
(RES/"fs03.json").write_text(json.dumps(fs03,indent=2)); PROGRESS["FS03"]="ok"
# keep model for FS04 comparison baseline
showtell_model=model
print("FS03 DONE")


## FS04 · Show Attend Tell


In [ ]:
# Show Attend Tell: spatial feature map + soft attention
class AttnEncoder(nn.Module):
    def __init__(self, emb=64):
        super().__init__()
        self.cnn=nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,emb,3,padding=1), nn.ReLU(),  # keep spatial [B,emb,H,W]
        )
    def forward(self,x):
        f=self.cnn(x)  # [B,C,h,w]
        B,C,h,w=f.shape
        return f.flatten(2).transpose(1,2), f  # [B,hw,C], map

class AttnDecoder(nn.Module):
    def __init__(self, emb=64, hid=128):
        super().__init__()
        self.emb=nn.Embedding(V, emb, padding_idx=stoi[PAD])
        self.attn_w=nn.Linear(emb+hid, 1)
        self.lstm=nn.LSTMCell(emb+emb, hid)
        self.fc=nn.Linear(hid, V)
        self.hid0=nn.Linear(emb, hid)
    def step(self, token, h, c, feats):
        # feats [B,L,C]
        B,L,C=feats.shape
        h_exp=h.unsqueeze(1).expand(-1,L,-1)
        e=self.attn_w(torch.cat([feats,h_exp],-1)).squeeze(-1)  # [B,L]
        a=torch.softmax(e, dim=-1)
        ctx=(a.unsqueeze(-1)*feats).sum(1)
        te=self.emb(token)
        h2,c2=self.lstm(torch.cat([te,ctx],-1),(h,c))
        return self.fc(h2), h2, c2, a
    def forward(self, feats, caps):
        B=feats.size(0); mean=feats.mean(1)
        h=torch.tanh(self.hid0(mean)); c=torch.zeros_like(h)
        logits=[]; attns=[]
        for t in range(caps.size(1)-1):
            lg,h,c,a=self.step(caps[:,t], h, c, feats)
            logits.append(lg); attns.append(a)
        return torch.stack(logits,1), torch.stack(attns,1)

class ShowAttend(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc=AttnEncoder(); self.dec=AttnDecoder()
    def forward(self,x,caps):
        feats,_=self.enc(x)
        return self.dec(feats, caps)

am=ShowAttend().to(device)
opt=torch.optim.Adam(am.parameters(), lr=2e-3)
hist4=[]
for epoch in range(1,12):
    am.train(); perm=torch.randperm(len(xtr)); losses=[]
    for i in range(0,len(xtr),64):
        b=perm[i:i+64]; xb,yb=xtr[b].to(device), ytr[b].to(device)
        opt.zero_grad(set_to_none=True)
        logits,att=am(xb,yb)
        loss=F.cross_entropy(logits.reshape(-1,V), yb[:,1:].reshape(-1), ignore_index=stoi[PAD])
        loss.backward(); opt.step(); losses.append(loss.item())
    am.eval()
    with torch.no_grad():
        lg,_=am(xva.to(device), yva.to(device))
        pred=lg.argmax(-1); mask=yva[:,1:].to(device)!=stoi[PAD]
        acc=(pred[mask]==yva[:,1:].to(device)[mask]).float().mean().item()
    row={"epoch":epoch,"loss":round(float(np.mean(losses)),4),"val_token_acc":round(acc,4)}
    hist4.append(row); print("FS04", row)

@torch.no_grad()
def greedy_attn(img_kind, max_len=8):
    img=make_shape_image(img_kind,64)[::2,::2][:32,:32]
    x=torch.tensor(img.transpose(2,0,1)[None],dtype=torch.float32,device=device)
    am.eval(); feats, fmap = am.enc(x)
    mean=feats.mean(1); h=torch.tanh(am.dec.hid0(mean)); c=torch.zeros_like(h)
    tok=torch.tensor([stoi[BOS]],device=device); ids=[]; maps=[]
    for t in range(max_len-1):
        lg,h,c,a=am.dec.step(tok, h, c, feats)
        nxt=lg.argmax(-1); maps.append(a.cpu().numpy()[0])
        if nxt.item()==stoi[EOS]: break
        ids.append(nxt.item()); tok=nxt
    words=[itos[i] for i in ids if i not in (stoi[PAD],stoi[BOS],stoi[EOS])]
    return " ".join(words), maps, fmap

samples4=[]; attn_vis=[]
fig,axes=plt.subplots(2,3,figsize=(9,5.5))
for j,k in enumerate(CLASSES):
    pred, maps, fmap = greedy_attn(k)
    samples4.append({"input":k,"pred":pred})
    axes[0,j].imshow(make_shape_image(k)); axes[0,j].set_title(f"pred: {pred}",fontsize=8); axes[0,j].axis("off")
    if maps:
        a=maps[min(1,len(maps)-1)]  # second token attn if any
        side=int(math.sqrt(len(a)))
        amap=a.reshape(side,side)
        axes[1,j].imshow(make_shape_image(k)); axes[1,j].imshow(amap, extent=[0,64,64,0], alpha=0.55, cmap="hot")
        axes[1,j].set_title("attn map",fontsize=8); axes[1,j].axis("off")
    else:
        axes[1,j].axis("off")
fig.suptitle("FS04 Show Attend Tell")
fig.tight_layout(); fig.savefig(FIG/"fs04_attention.png",dpi=120); plt.close()

# compare token acc
cmp={"fs03_final_token_acc":hist[-1]["val_token_acc"],"fs04_final_token_acc":hist4[-1]["val_token_acc"],
     "fs03_samples":samples,"fs04_samples":samples4}
print(cmp)
fs04={"stage":"FS04","method":"Show-Attend-Tell soft attention over spatial CNN map",
      "history":hist4,"samples":samples4,"compare":cmp,
      "vs_prev":"FS03 single global vector; FS04 looks at different regions per word",
      "figure":"figures/fs04_attention.png"}
(RES/"fs04.json").write_text(json.dumps(fs04,indent=2)); PROGRESS["FS04"]="ok"
print("FS04 DONE")


In [ ]:
summary={"notebook":"Grok-multimodal-fs03-fs04-captioning","progress":PROGRESS,
         "device":str(device),"gpu_count":torch.cuda.device_count() if torch.cuda.is_available() else 0}
(RES/"summary_fs03_fs04.json").write_text(json.dumps(summary,indent=2))
(OUT/"SUCCESS").write_text("ok\n"); print(summary); print("FS03-04 COMPLETE")
